# LLM Data Format Converter

Convert between `json`, `yaml`, `toml`, `ini`, `hcl`, and `xml` using an LLM.

## What this app does
- Auto-detects the **input** format.
- Lets you choose the **target** format.
- Uses an expert **system prompt** to guide reliable, schema-preserving conversion.
- Validates generated output where possible.

## Setup
1. Install dependencies (if needed):
   - `pip install openai gradio python-dotenv pyyaml toml`
2. Add this key to `.env`:
   - `OPENROUTER_API_KEY=...`
3. OpenRouter base URL is hardcoded in the notebook as:
   - `https://openrouter.ai/api/v1`
4. Run cells top-to-bottom, then launch the Gradio app cell.

In [ ]:
import os
import re
import json
import configparser
import xml.etree.ElementTree as ET
from typing import Tuple

import gradio as gr
import yaml
from dotenv import load_dotenv
from openai import OpenAI

# Python 3.11+ has tomllib in stdlib. Fall back to toml package if needed.
try:
    import tomllib  
except Exception:
    tomllib = None
    import toml 

load_dotenv(override=True)

SUPPORTED_FORMATS = ["json", "yaml", "toml", "ini", "hcl", "xml"]
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "openai/gpt-4o-mini"

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_client = None
if openrouter_api_key:
    openrouter_client = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL)
    print(f"OpenRouter client ready with model: {OPENROUTER_MODEL}")
else:
    print("OPENROUTER_API_KEY missing. Add it to .env to run conversions.")

In [ ]:
def _parse_json(text: str) -> bool:
    """Check whether input parses as JSON.

    Args:
        text: Raw input payload.

    Returns:
        True if the payload is valid JSON, otherwise False.
    """
    try:
        json.loads(text)
        return True
    except Exception:
        return False


def _parse_yaml(text: str) -> bool:
    """Check whether input parses as structured YAML.

    Args:
        text: Raw input payload.

    Returns:
        True if payload parses as YAML mapping/list, otherwise False.
    """
    try:
        parsed = yaml.safe_load(text)
        # Avoid classifying arbitrary scalar strings as YAML.
        if isinstance(parsed, (dict, list)):
            return True
        return False
    except Exception:
        return False


def _parse_toml(text: str) -> bool:
    """Check whether input parses as TOML.

    Args:
        text: Raw input payload.

    Returns:
        True if the payload is valid TOML, otherwise False.
    """
    try:
        if tomllib:
            tomllib.loads(text)
        else:
            toml.loads(text)
        return True
    except Exception:
        return False


def _parse_ini(text: str) -> bool:
    """Check whether input parses as INI with at least one section.

    Args:
        text: Raw input payload.

    Returns:
        True if payload parses as INI and includes section headers, otherwise False.
    """
    try:
        parser = configparser.ConfigParser()
        parser.read_string(text)
        # Require at least one section to reduce false positives.
        return len(parser.sections()) > 0
    except Exception:
        return False


def _parse_xml(text: str) -> bool:
    """Check whether input parses as XML.

    Args:
        text: Raw input payload.

    Returns:
        True if payload is valid XML, otherwise False.
    """
    try:
        ET.fromstring(text)
        return True
    except Exception:
        return False


def _looks_like_hcl(text: str) -> bool:
    """Detect whether input resembles HCL using heuristics.

    Args:
        text: Raw input payload.

    Returns:
        True if payload matches common HCL block/assignment patterns, otherwise False.
    """
    stripped = text.strip()
    if not stripped:
        return False

    has_block = bool(re.search(r"^\s*[A-Za-z_][\w-]*\s+\"[^\"]+\"(?:\s+\"[^\"]+\")?\s*\{", stripped, re.MULTILINE))
    has_assignment = bool(re.search(r"^\s*[A-Za-z_][\w-]*\s*=\s*.+$", stripped, re.MULTILINE))
    has_braces = "{" in stripped and "}" in stripped

    return (has_block and has_braces) or (has_assignment and has_braces)


def detect_input_format(text: str) -> Tuple[str, str, str]:
    """Infer the most likely source format from the input payload.

    Args:
        text: Raw input payload.

    Returns:
        A tuple of `(format_name, confidence, reason)` where confidence is one of
        `high`, `medium`, or `low`.
    """
    payload = (text or "").strip()
    if not payload:
        return "unknown", "low", "Input is empty"

    # Order matters: JSON before YAML because YAML is a superset.
    if _parse_json(payload):
        return "json", "high", "Valid JSON parse"

    # TOML and INI before YAML to reduce YAML over-detection.
    if _parse_toml(payload) and "=" in payload:
        return "toml", "high", "Valid TOML parse"

    if _parse_ini(payload):
        return "ini", "high", "Valid INI parse with section headers"

    if _parse_xml(payload):
        return "xml", "high", "Valid XML parse"

    if _looks_like_hcl(payload):
        return "hcl", "medium", "Matches common HCL block/assignment patterns"

    if _parse_yaml(payload):
        return "yaml", "medium", "Valid YAML parse (mapping/list)"

    return "unknown", "low", "No supported format confidently detected"

In [ ]:
CONVERTER_SYSTEM_PROMPT = """You are a senior data serialization engineer.

You convert data between JSON, YAML, TOML, INI, HCL, and XML with strict correctness.
Rules:
1) Preserve meaning exactly: keys, nesting, arrays/lists, booleans, numbers, null-like values, and strings.
2) Output ONLY the converted data payload in the requested target format.
3) Do not include markdown fences, comments, explanations, or extra text.
4) Produce syntactically valid target format.
5) If the input is ambiguous, infer the most likely intent while preserving original values.
"""


def build_conversion_user_prompt(source_format: str, target_format: str, input_text: str) -> str:
    """Build the user prompt for format conversion.

    Args:
        source_format: Detected source format name.
        target_format: Desired output format name.
        input_text: Raw input payload to convert.

    Returns:
        A formatted user prompt string with conversion constraints.
    """
    return f"""Convert the following data from {source_format} to {target_format}.

Constraints:
- Keep semantic equivalence.
- Keep all fields; do not drop or invent data.
- Ensure target syntax is valid for {target_format}.
- Return only the converted payload.

Input:
{input_text}
"""

In [ ]:
def strip_markdown_fences(text: str) -> str:
    """Remove optional markdown code fences from model output.

    Args:
        text: Raw text returned by the LLM.

    Returns:
        Cleaned text without leading/trailing fenced-code markers.
    """
    cleaned = re.sub(r"^```[a-zA-Z0-9_-]*\n", "", text.strip())
    cleaned = re.sub(r"\n```$", "", cleaned)
    return cleaned.strip()


def validate_as_format(text: str, fmt: str) -> Tuple[bool, str]:
    """Validate converted output against the requested target format.

    Args:
        text: Converted payload produced by the model.
        fmt: Target format name.

    Returns:
        A tuple of `(is_valid, message)` describing validation status.
    """
    try:
        if fmt == "json":
            json.loads(text)
            return True, "Valid JSON"
        if fmt == "yaml":
            yaml.safe_load(text)
            return True, "Valid YAML"
        if fmt == "toml":
            if tomllib:
                tomllib.loads(text)
            else:
                toml.loads(text)
            return True, "Valid TOML"
        if fmt == "ini":
            parser = configparser.ConfigParser()
            parser.read_string(text)
            if len(parser.sections()) == 0:
                return False, "INI parsed but no sections found"
            return True, "Valid INI"
        if fmt == "xml":
            ET.fromstring(text)
            return True, "Valid XML"
        if fmt == "hcl":
            if _looks_like_hcl(text):
                return True, "Looks like HCL (heuristic check)"
            return False, "Could not verify HCL heuristically"
        return False, f"Unsupported target format: {fmt}"
    except Exception as exc:
        return False, f"Validation failed for {fmt}: {exc}"


def convert_data_with_llm(
    input_data: str,
    target_format: str,
    normalize_if_same_format: bool,
) -> Tuple[str, str]:
    """Convert input payload to target format using OpenRouter.

    Args:
        input_data: Raw source payload from the UI.
        target_format: Desired output format.
        normalize_if_same_format: Whether to still run model conversion when
            detected source and target formats match.

    Returns:
        A tuple of `(converted_output, metadata_markdown)`.
    """
    if not input_data or not input_data.strip():
        return "", "Please provide input data."

    if target_format not in SUPPORTED_FORMATS:
        return "", f"Unsupported target format: {target_format}"

    if not openrouter_client:
        return "", "OpenRouter client not configured. Add OPENROUTER_API_KEY to .env"

    source_format, confidence, reason = detect_input_format(input_data)

    if source_format == target_format and not normalize_if_same_format:
        meta = (
            f"### Conversion Metadata\n"
            f"- Detected source: `{source_format}` ({confidence})\n"
            f"- Target format: `{target_format}`\n"
            f"- Provider: `OpenRouter`\n"
            f"- Model: `{OPENROUTER_MODEL}`\n"
            f"- Detection reason: {reason}\n"
            f"- Result: Source and target are the same; returning original input."
        )
        return input_data.strip(), meta

    user_prompt = build_conversion_user_prompt(source_format, target_format, input_data)

    try:
        response = openrouter_client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=[
                {"role": "system", "content": CONVERTER_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
        )
        raw = response.choices[0].message.content or ""
        converted = strip_markdown_fences(raw)

        is_valid, validation_message = validate_as_format(converted, target_format)

        meta = (
            f"### Conversion Metadata\n"
            f"- Detected source: `{source_format}` ({confidence})\n"
            f"- Target format: `{target_format}`\n"
            f"- Provider: `OpenRouter`\n"
            f"- Model: `{OPENROUTER_MODEL}`\n"
            f"- Detection reason: {reason}\n"
            f"- Validation: {'PASS' if is_valid else 'WARN'} - {validation_message}"
        )

        return converted, meta

    except Exception as exc:
        return "", (
            f"### Conversion Error\n"
            f"- Provider: `OpenRouter`\n"
            f"- Model: `{OPENROUTER_MODEL}`\n"
            f"- Error: `{exc}`\n"
            f"- Tip: Check OPENROUTER_API_KEY, model name, and network connectivity."
        )

In [ ]:
EXAMPLE_JSON = '''{
  "service": "orders",
  "replicas": 3,
  "enabled": true,
  "tags": ["api", "prod"],
  "database": {
    "host": "localhost",
    "port": 5432
  }
}'''

EXAMPLE_XML = '''<config>
  <service>orders</service>
  <replicas>3</replicas>
  <enabled>true</enabled>
  <database>
    <host>localhost</host>
    <port>5432</port>
  </database>
</config>'''

EXAMPLE_HCL = '''resource "aws_instance" "web" {
  ami           = "ami-0abcdef1234567890"
  instance_type = "t3.micro"
  tags = {
    Name = "web-server"
    Env  = "dev"
  }
}'''


def create_ui() -> gr.Blocks:
    """Create the Gradio interface for data format conversion.

    Returns:
        A configured Gradio `Blocks` app instance.
    """
    with gr.Blocks(title="LLM Data Format Converter", theme=gr.themes.Soft()) as demo:
        gr.Markdown(
            """
            # LLM Data Format Converter

            Convert between **json, yaml, toml, ini, hcl, xml**.
            - Input format is auto-detected.
            - Output format is selected by you.
            - LLM conversion is guided by an expert system prompt.
            - Provider is fixed to OpenRouter.
            """
        )

        with gr.Row():
            with gr.Column(scale=2):
                input_data = gr.Textbox(
                    label="Input Data",
                    placeholder="Paste source data here...",
                    lines=18,
                )

                target_format = gr.Dropdown(
                    choices=SUPPORTED_FORMATS,
                    value="yaml",
                    label="Target Format",
                )

                normalize_same = gr.Checkbox(
                    value=False,
                    label="If source and target are same, still normalize using LLM",
                )

                convert_btn = gr.Button("Convert", variant="primary")

            with gr.Column(scale=2):
                output_data = gr.Textbox(label="Converted Output", lines=18)
                metadata = gr.Markdown(label="Metadata")

        convert_btn.click(
            fn=convert_data_with_llm,
            inputs=[input_data, target_format, normalize_same],
            outputs=[output_data, metadata],
        )

        gr.Examples(
            examples=[
                [EXAMPLE_JSON, "xml", False],
                [EXAMPLE_XML, "json", False],
                [EXAMPLE_HCL, "yaml", False],
            ],
            inputs=[input_data, target_format, normalize_same],
        )

    return demo

## Run the app

Execute the next cell to launch Gradio.

Notes:
- HCL detection and validation are heuristic because the standard library does not include a strict HCL parser.
- YAML can parse many shapes, so parser order is tuned to avoid YAML over-matching.
- For production-critical transformations, keep a post-conversion parser validation step.

In [ ]:
# Quick detection smoke tests (optional)
smoke_samples = {
    "json": '{"name": "alice", "age": 30}',
    "yaml": 'name: alice\nage: 30',
    "toml": 'name = "alice"\nage = 30',
    "ini": '[user]\nname=alice\nage=30',
    "xml": '<user><name>alice</name><age>30</age></user>',
    "hcl": 'resource "x" "y" {\n  name = "alice"\n}',
}

for expected, sample in smoke_samples.items():
    detected, conf, reason = detect_input_format(sample)
    print(f"expected={expected:<4} detected={detected:<7} confidence={conf:<6} reason={reason}")

In [ ]:
demo = create_ui()
demo.launch(inbrowser=True, share=False)